# Illicit Bitcoin transaction detection (Elliptic) — final model

This notebook reproduces the final Kaggle submission end to end: data checks, feature
preparation, time-forward validation, final training and the submission file. The
experiments that led to these choices are run by the scripts in `src/` and recorded in
`report/findings.md`; their saved result tables are summarised at the end.

**Approach in brief**
* Supervised training uses only transactions with a confirmed label (`0` licit / `1` illicit).
  `none` rows are unlabeled and excluded from the supervised fit.
* `time_step` is used only to split data chronologically, never as a model input; trees cannot
  extrapolate a raw time counter into the unseen test steps 36-49.
* Validation always trains on earlier steps and scores later steps, over several windows and
  random seeds, because single-seed, single-split scores varied by up to 0.02 ROC-AUC.
* Labeled transactions cluster: 57% of an illicit transaction's labeled neighbours are illicit
  (base rate 12%). The final model adds graph context with a two-stage stack: stage-1 scores
  and features of *labeled* neighbours (one and two hops away) become features for stage 2.
  Context is restricted to labeled nodes because those are the only transactions with features
  in the test period, so every graph feature is computed the same way in training and test.
* Training rows are weighted toward recent steps (half-life 14 steps).
* Both stages use a tuned, shallow LightGBM; the configuration is in `src/final_config.py`
  (Kaggle public ROC-AUC 0.96668).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data import load_data
from src.evaluation import REPORT_WINDOWS, labeled_frame, window_table
from src.features import get_feature_groups
from src.final_config import FINAL_NAME, FINAL_SEEDS, FINAL_STACK
from src.graph import audit_graph_coverage
from src.graph_stacking import build_graph, two_hop_features
from src import train as final_pipeline

pd.set_option("display.width", 160)

## 1. Data audit

`train.csv` holds steps 1-35 (labeled and unlabeled rows), `test.csv` steps 36-49, and
`txs_edgelist.csv` the payment-flow edges (an output of `txId1` is spent by `txId2`).
`load_data` validates the schema, label values, unique IDs and numeric features.

In [2]:
train, test, edges, sample_submission = load_data(PROJECT_ROOT / "data")
labeled = labeled_frame(train)
groups = get_feature_groups(train.columns)

print(f"train rows: {len(train):,} | test rows: {len(test):,} | edges: {len(edges):,}")
print("labels:", train["label"].value_counts(dropna=False).rename({pd.NA: "none"}).to_dict())
print("missing feature values:", int(train[groups['all']].isna().sum().sum() + test[groups['all']].isna().sum().sum()))
print(f"labeled train + test = {len(labeled) + len(test):,} (the documented Elliptic labeled total is 46,564)")

train rows: 141,772 | test rows: 15,329 | edges: 234,355
labels: {'none': 110537, 0: 27591, 1: 3644}
missing feature values: 0
labeled train + test = 46,564 (the documented Elliptic labeled total is 46,564)


In [3]:
per_step = labeled.groupby("time_step")["label"].agg(labeled_rows="size", illicit="sum")
per_step["illicit_share"] = (per_step["illicit"] / per_step["labeled_rows"]).round(3)
per_step.T

time_step,1,2,3,4,5,6,7,8,9,10,...,26,27,28,29,30,31,32,33,34,35
labeled_rows,2147.000,1117.000,1279.000,1440.000,1882.000,485.00,1203.000,1165.000,778.000,972.000,...,517.000,206.000,284.000,1174.00,524.000,710.000,1323.000,441.000,515.000,1341.000
illicit,17.000,18.000,11.000,30.000,8.000,5.00,102.000,67.000,248.000,18.000,...,96.000,24.000,85.000,329.00,83.000,106.000,342.000,23.000,37.000,182.000
illicit_share,0.008,0.016,0.009,0.021,0.004,0.01,0.085,0.058,0.319,0.019,...,0.186,0.117,0.299,0.28,0.158,0.149,0.259,0.052,0.072,0.136


The test file contains exactly the labeled transactions of steps 36-49. The edge list also
references 46,668 transactions with no feature row: these are the unlabeled transactions of
steps 36-49. No edge connects two different time steps.

In [4]:
audit = audit_graph_coverage(train, test, edges)
time_of = pd.concat([train.set_index("txId")["time_step"], test.set_index("txId")["time_step"]])
dedup = edges.drop_duplicates()
src_t, dst_t = dedup["txId1"].map(time_of), dedup["txId2"].map(time_of)
known = src_t.notna() & dst_t.notna()
audit["cross_time_step_edges"] = int((src_t[known] != dst_t[known]).sum())
test_ids = set(test["txId"]); feature_ids = set(train["txId"]) | test_ids
touch_test = dedup[dedup["txId1"].isin(test_ids) | dedup["txId2"].isin(test_ids)]
other = np.where(touch_test["txId1"].isin(test_ids), touch_test["txId2"], touch_test["txId1"])
audit["share_of_test_edges_to_featureless_nodes"] = round(float((~pd.Series(other).isin(feature_ids)).mean()), 3)
pd.Series(audit)

num_train_nodes                             141772.000000
num_test_nodes                               15329.000000
num_feature_nodes                           157101.000000
num_edge_endpoints                          203769.000000
num_union_nodes                             203769.000000
num_missing_feature_nodes                    46668.000000
num_orphan_feature_nodes                         0.000000
num_total_edges                             234355.000000
num_edges_induced                           175918.000000
num_edges_lost_induced                       58437.000000
pct_edges_lost_induced                          24.935248
cross_time_step_edges                            0.000000
share_of_test_edges_to_featureless_nodes         0.656000
dtype: float64

Two-thirds of a test transaction's neighbours have no features, while every training
transaction's neighbours do. A graph neural network that reads all neighbour features
therefore meets a different input distribution at test time (GraphSAGE lowered the Kaggle
score; `report/findings.md` sections 1.3 and 7). The final model avoids this by using only
*labeled* neighbours, which have features in both periods, and by letting two-hop walks pass
through featureless nodes. Coverage of that context is similar in training and test:

In [5]:
graph = build_graph(edges)
context_rows = pd.concat([labeled[["txId"]], test[["txId"]]], ignore_index=True)
scored = set(context_rows["txId"])
one_hop = graph[graph["u"].isin(scored) & graph["v"].isin(scored)].groupby("u").size()
dummy = context_rows.assign(**{c: 0.0 for c in ["feat_3", "feat_5", "feat_53", "feat_59", "feat_61"]})
two_hop = two_hop_features(graph, dummy, pd.Series(0.0, index=context_rows["txId"]))["hop2_labeled_walks"]
coverage = {}
for name, frame in [("labeled training rows", labeled), ("test rows", test)]:
    has_one = one_hop.reindex(frame["txId"]).fillna(0).to_numpy() > 0
    has_two = two_hop.reindex(frame["txId"]).fillna(0).to_numpy() > 0
    coverage[name] = {"1-hop labeled neighbour": has_one.mean(), "2-hop labeled node": has_two.mean(),
                      "2-hop only": (has_two & ~has_one).mean(), "no graph context": (~has_one & ~has_two).mean()}
pd.DataFrame(coverage).T.round(3)

,1-hop labeled neighbour,2-hop labeled node,2-hop only,no graph context
labeled training rows,0.782,0.878,0.143,0.075
test rows,0.746,0.938,0.218,0.036


## 2. Features and distribution shift

Per the Elliptic documentation, `feat_1`-`feat_93` describe the transaction itself (local) and
`feat_94`-`feat_165` aggregate its one-hop neighbourhood. The table measures how far each group's
medians move, in units of the earlier period's interquartile range.

In [6]:
def median_shift(reference, target, columns):
    iqr = (reference[columns].quantile(0.75) - reference[columns].quantile(0.25)).replace(0, np.nan)
    return ((target[columns].median() - reference[columns].median()).abs() / iqr).fillna(0)

rows = []
comparisons = {
    "steps<=28 -> 29-35 (validation)": (labeled[labeled.time_step <= 28], labeled[labeled.time_step > 28]),
    "steps<=35 -> test 36-49": (labeled, test),
}
for name, (reference, target) in comparisons.items():
    for group in ("local", "aggregated"):
        shift = median_shift(reference, target, groups[group])
        rows.append({"comparison": name, "group": group, "features": len(groups[group]),
                     "mean_shift_IQR": round(shift.mean(), 3), "share_shift_over_1_IQR": round((shift > 1).mean(), 3)})
pd.DataFrame(rows)

,comparison,group,features,mean_shift_IQR,share_shift_over_1_IQR
0,steps<=28 -> 29-35 (validation),local,93,0.050,0.000
1,steps<=28 -> 29-35 (validation),aggregated,72,0.281,0.097
2,steps<=35 -> test 36-49,local,93,0.053,0.011
3,steps<=35 -> test 36-49,aggregated,72,0.439,0.194


The aggregated features drift about twice as much into the test period as within the training
period. A model built only on aggregated features scored 0.880 on Kaggle versus 0.943 for all
features, so the final model keeps all supplied features and relies on regularisation rather
than on the aggregated group.

## 3. Final model configuration

`graph_stacking.make_neighbour_stack(**FINAL_STACK)`:
1. **Stage 1** — tuned LightGBM on the 165 supplied features scores labeled training rows
   out-of-fold (5 folds of consecutive time steps) and test rows with the mean of the 5 fold
   models.
2. **Graph context** — for every transaction: mean / max / incoming-mean / outgoing-mean of
   stage-1 scores over labeled neighbours; in/out degree; number and share of labeled
   neighbours; neighbour means of feat_53, feat_59, feat_61, feat_5, feat_3; and walk-weighted
   two-hop means of the stage-1 score and those five features.
3. **Stage 2** — the same LightGBM on supplied features + graph context.
4. Both stages weight training rows by 2^((t - last step)/14); classes are balanced; seeds
   0-9 are averaged.

In [7]:
from src.model_zoo import TUNED_LGBM_PARAMS
print("final model:", FINAL_NAME, "| stack options:", FINAL_STACK, "| seeds:", len(FINAL_SEEDS))
pd.Series(TUNED_LGBM_PARAMS, name="LightGBM settings (both stages)")

final model: nbr_stack_rich_2hop_hl14 | stack options: {'half_life_steps': 14, 'rich': True, 'two_hop': True} | seeds: 10


n_estimators         500.000000
learning_rate          0.025268
num_leaves             9.000000
min_child_samples     49.000000
colsample_bytree       0.644457
subsample              0.982643
subsample_freq         1.000000
reg_lambda             2.035594
Name: LightGBM settings (both stages), dtype: float64

## 4. Time-forward validation

Each window trains on labeled rows up to a step and scores a later block. Three seeds per window
are averaged (seed bagging, as in the submission). The reference is the tuned LightGBM without
graph context (Kaggle 0.94417). The short windows (train <= 25 or <= 28) are the ones whose
model ranking matched the Kaggle leaderboard; the graph stack wins there but loses on the long
windows, and the Kaggle leaderboard (+0.0225 over the reference) sided with the short windows.
Local windows cannot reproduce the test period's drift, so they serve here as a reproducibility
check rather than as the selection criterion (`report/findings.md` sections 11, 14, 20).
Runtime: several minutes.

In [8]:
tables = final_pipeline.validate(labeled, graph, seeds=(0, 1, 2), output_dir=PROJECT_ROOT / "artifacts" / "final")
window_table(tables["window_metrics"]).round(4)

window,t14_v15-35,t21_v22-28,t21_v22-35,t25_v26-35,t28_v29-35,mean
candidate,,,,,,
tuned_lgbm_all,0.9010,0.9583,0.9109,0.9954,0.9960,0.9523
nbr_stack_rich_2hop_hl14,0.8943,0.9453,0.8783,0.9970,0.9983,0.9426


In [9]:
tables["summary"].round(4)

,candidate,mean_roc_auc,worst_roc_auc,mean_delta_vs_baseline,min_window_delta,windows_improved,windows,mean_seed_auc_std,adopt
0,tuned_lgbm_all,0.9523,0.9010,0.0000,0.0000,0,5,0.0019,False
1,nbr_stack_rich_2hop_hl14,0.9426,0.8783,-0.0097,-0.0326,2,5,0.0033,False


## 5. Final training and submission

Both stages are refitted on all 31,235 labeled rows from steps 1-35 for each of the ten seeds,
and the seed predictions for the test rows are averaged. The writer enforces the `index,target`
format and the sample-submission row order.

In [10]:
target = final_pipeline.stack_predict(labeled, test, graph, FINAL_SEEDS)
submission = pd.DataFrame({"index": test["index"].to_numpy(), "target": target})
path = final_pipeline.write_submission(submission, PROJECT_ROOT / "submissions" / "submission_final.csv",
                                       sample_submission=sample_submission)
print(path.relative_to(PROJECT_ROOT), submission.shape)
submission.describe().T

submissions/submission_final.csv (15329, 2)


,count,mean,std,min,25%,50%,75%,max
index,15329.0,7664.000000,4425.245473,0.000000,3832.000000,7664.000000,11496.000000,15328.000000
target,15329.0,0.062555,0.201632,0.000049,0.000609,0.002122,0.015374,0.999258


## 6. Experiment record

Summary tables saved by the experiment scripts (if present locally). Full commentary, including
the Kaggle scores of each submitted variant, is in `report/findings.md`.

In [11]:
for label, relative in [
    ("Step 1 - harness comparison", "artifacts/harness/step1_summary.csv"),
    ("Step 4 - tuned models and ensembles", "artifacts/tuning/step4_summary.csv"),
    ("Step 5 - stress tests", "artifacts/stress/stress_summary.csv"),
    ("Step 6 - GraphSAGE", "artifacts/extensions/gnn_summary.csv"),
    ("Step 6 - pseudo-labeling", "artifacts/extensions/pseudo_summary.csv"),
    ("Rounds 3-5 - graph stacking variants (local check)", "artifacts/round3/round5_window_table.csv"),
]:
    path = PROJECT_ROOT / relative
    if path.exists():
        print(f"\n=== {label} ({relative}) ===")
        print(pd.read_csv(path).round(4).head(15).to_string(index=False))


=== Step 1 - harness comparison (artifacts/harness/step1_summary.csv) ===
            candidate  mean_roc_auc  worst_roc_auc  mean_delta_vs_baseline  min_window_delta  windows_improved  windows  mean_seed_auc_std  adopt
blend_shallow_all_agg        0.9606         0.9258                  0.0187           -0.0103                 3        5             0.0020  False
          shallow_all        0.9467         0.8890                  0.0047           -0.0090                 2        5             0.0032   True
          shallow_agg        0.9462         0.9213                  0.0042           -0.0432                 3        5             0.0016  False
         baseline_all        0.9419         0.8770                  0.0000            0.0000                 0        5             0.0039  False
         baseline_agg        0.9374         0.8994                 -0.0046           -0.0445                 2        5             0.0015  False
        shallow_local        0.9364         0.873